# Module 4: Calculus — Multivariable

Neural networks have **millions of parameters**. We need calculus that works with functions of many variables.

### 🎯 What you'll learn:
- Partial derivatives
- Gradients and directional derivatives
- Jacobian matrices
- Hessian matrices
- Lagrange multipliers (constrained optimization)
- Multiple integrals

### 🤖 Why it matters for AI:
- **Gradient** = direction of steepest ascent (negate for descent!)
- **Jacobian** = how a vector function changes w.r.t. inputs (used in backprop)
- **Hessian** = curvature of loss surface (Newton's method, Adam)
- **Lagrange multipliers** = constrained optimization (SVMs, regularization)

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

x, y, z = sp.symbols('x y z')
plt.rcParams['figure.figsize'] = (10, 6)
plt.style.use('seaborn-v0_8-darkgrid')

---
## 1. Partial Derivatives

For $f(x, y)$, the **partial derivative** with respect to $x$ treats $y$ as a constant:

$$\frac{\partial f}{\partial x} = \lim_{h \to 0} \frac{f(x+h, y) - f(x, y)}{h}$$

### Example: $f(x,y) = x^2y + \sin(xy)$
$$\frac{\partial f}{\partial x} = 2xy + y\cos(xy) \qquad \frac{\partial f}{\partial y} = x^2 + x\cos(xy)$$

### 🤖 AI Connection:
When training a neural network, we compute $\frac{\partial L}{\partial w_i}$ for **each** weight $w_i$ — these are all partial derivatives!

In [ ]:
# Partial derivatives
f = x**2 * y + sp.sin(x * y)
print(f"f(x,y) = {f}")
print(f"∂f/∂x = {sp.diff(f, x)}")
print(f"∂f/∂y = {sp.diff(f, y)}")

# Numerical partial derivatives
def numerical_partial(f, point, var_idx, h=1e-7):
    """Compute partial derivative of f at point w.r.t. variable var_idx."""
    point_plus = point.copy()
    point_minus = point.copy()
    point_plus[var_idx] += h
    point_minus[var_idx] -= h
    return (f(*point_plus) - f(*point_minus)) / (2 * h)

f_np = lambda x, y: x**2 * y + np.sin(x * y)
point = [2.0, 3.0]
print(f"\nAt (2, 3):")
print(f"  ∂f/∂x ≈ {numerical_partial(f_np, point, 0):.6f}")
print(f"  ∂f/∂y ≈ {numerical_partial(f_np, point, 1):.6f}")

---
## 2. The Gradient — Direction of Steepest Ascent

The **gradient** of $f: \mathbb{R}^n \to \mathbb{R}$ is the vector of all partial derivatives:

$$\nabla f = \begin{bmatrix} \frac{\partial f}{\partial x_1} \\ \frac{\partial f}{\partial x_2} \\ \vdots \\ \frac{\partial f}{\partial x_n} \end{bmatrix}$$

### Key Properties:
1. $\nabla f$ points in the direction of **steepest ascent**
2. $-\nabla f$ points in the direction of **steepest descent** (used in gradient descent!)
3. $\|\nabla f\|$ gives the **rate** of steepest ascent
4. $\nabla f$ is **perpendicular** to level curves

In [ ]:
# Gradient visualization
f_np = lambda x, y: x**2 + y**2  # Simple bowl

# Compute gradient at several points
x_vals = np.linspace(-3, 3, 15)
y_vals = np.linspace(-3, 3, 15)
X, Y = np.meshgrid(x_vals, y_vals)
Z = f_np(X, Y)

# Gradient: ∇f = [2x, 2y]
dfdx = 2 * X
dfdy = 2 * Y

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Contour plot with gradient vectors
cs = axes[0].contour(X, Y, Z, levels=15, cmap='viridis')
axes[0].clabel(cs, inline=True, fontsize=8)
axes[0].quiver(X, Y, -dfdx, -dfdy, alpha=0.6, color='red')  # Negative gradient
axes[0].set_title('Contours + Negative Gradient (Descent Direction)', fontsize=13)
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
axes[0].set_aspect('equal')

# 3D surface
ax3d = fig.add_subplot(122, projection='3d')
X2, Y2 = np.meshgrid(np.linspace(-3, 3, 50), np.linspace(-3, 3, 50))
ax3d.plot_surface(X2, Y2, f_np(X2, Y2), cmap='viridis', alpha=0.8)
ax3d.set_title('f(x,y) = x² + y²', fontsize=14)
ax3d.set_xlabel('x'); ax3d.set_ylabel('y'); ax3d.set_zlabel('f')

plt.tight_layout()
plt.show()

---
## 3. Directional Derivative

The **directional derivative** measures the rate of change of $f$ in direction $\mathbf{u}$:

$$D_{\mathbf{u}} f = \nabla f \cdot \mathbf{u} = \|\nabla f\| \cos\theta$$

Where $\mathbf{u}$ is a unit vector and $\theta$ is the angle between $\nabla f$ and $\mathbf{u}$.

### Key insight:
- Maximum when $\mathbf{u} \parallel \nabla f$ ($\theta = 0$) → steepest ascent
- Minimum when $\mathbf{u} \parallel -\nabla f$ ($\theta = \pi$) → steepest descent
- Zero when $\mathbf{u} \perp \nabla f$ ($\theta = \pi/2$) → along level curve

In [ ]:
# Directional derivative
f = x**2 + 3*y**2
grad_f = [sp.diff(f, x), sp.diff(f, y)]
print(f"f(x,y) = {f}")
print(f"∇f = {grad_f}")

# At point (1, 1), in direction u = [1/√2, 1/√2]
point = {x: 1, y: 1}
grad_at_point = np.array([float(g.subs(point)) for g in grad_f])
u = np.array([1/np.sqrt(2), 1/np.sqrt(2)])

dir_deriv = np.dot(grad_at_point, u)
print(f"\nAt (1,1): ∇f = {grad_at_point}")
print(f"Direction u = {np.round(u, 4)}")
print(f"Directional derivative = {dir_deriv:.4f}")
print(f"Max rate of change = ||∇f|| = {np.linalg.norm(grad_at_point):.4f}")

---
## 4. The Jacobian Matrix

For a vector-valued function $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$:

$$\mathbf{J} = \begin{bmatrix} \frac{\partial f_1}{\partial x_1} & \cdots & \frac{\partial f_1}{\partial x_n} \\ \vdots & \ddots & \vdots \\ \frac{\partial f_m}{\partial x_1} & \cdots & \frac{\partial f_m}{\partial x_n} \end{bmatrix}$$

The Jacobian is the **best linear approximation** of $\mathbf{f}$ near a point.

### 🤖 AI Connection:
- In backprop, we compute **Jacobian-vector products** (not full Jacobians)
- **Normalizing flows** use $|\det(J)|$ for change of variables
- The Jacobian relates input perturbations to output perturbations

In [ ]:
# Jacobian example: f(x,y) = [x²y, sin(x+y)]
f1 = x**2 * y
f2 = sp.sin(x + y)

J = sp.Matrix([[sp.diff(f1, x), sp.diff(f1, y)],
               [sp.diff(f2, x), sp.diff(f2, y)]])

print(f"f(x,y) = [{f1}, {f2}]")
print(f"\nJacobian:\n{J}")

# Evaluate at (1, π/2)
J_at_point = J.subs([(x, 1), (y, sp.pi/2)])
print(f"\nJ at (1, π/2):\n{J_at_point}")
print(f"det(J) = {J_at_point.det()}")

# Numerical Jacobian computation
def numerical_jacobian(f, point, h=1e-7):
    """Compute Jacobian numerically.
    f: function taking array, returning array
    point: numpy array
    """
    n = len(point)
    f0 = f(point)
    m = len(f0)
    J = np.zeros((m, n))
    for j in range(n):
        p_plus = point.copy()
        p_plus[j] += h
        J[:, j] = (f(p_plus) - f0) / h
    return J

f_np = lambda p: np.array([p[0]**2 * p[1], np.sin(p[0] + p[1])])
J_num = numerical_jacobian(f_np, np.array([1.0, np.pi/2]))
print(f"\nNumerical Jacobian:\n{np.round(J_num, 4)}")

---
## 5. The Hessian Matrix

The **Hessian** is the matrix of second partial derivatives:

$$\mathbf{H} = \begin{bmatrix} \frac{\partial^2 f}{\partial x_1^2} & \frac{\partial^2 f}{\partial x_1 \partial x_2} & \cdots \\ \frac{\partial^2 f}{\partial x_2 \partial x_1} & \frac{\partial^2 f}{\partial x_2^2} & \cdots \\ \vdots & \vdots & \ddots \end{bmatrix}$$

### The Hessian tells us about curvature:
- **Positive definite** $H$ → local minimum (convex bowl)
- **Negative definite** $H$ → local maximum
- **Indefinite** $H$ → saddle point

### 🤖 AI Connection:
- **Newton's method**: $\theta_{t+1} = \theta_t - H^{-1} \nabla f$ (uses curvature for faster convergence)
- **Adam optimizer** approximates diagonal Hessian information
- Saddle points (indefinite Hessian) are everywhere in high-dimensional loss surfaces

In [ ]:
# Hessian example
f = x**3 - 3*x*y**2 + y**3  # Monkey saddle
print(f"f(x,y) = {f}")

# Compute Hessian symbolically
H = sp.Matrix([[sp.diff(f, x, x), sp.diff(f, x, y)],
               [sp.diff(f, y, x), sp.diff(f, y, y)]])
print(f"\nHessian:\n{H}")

# Evaluate at origin
H_origin = H.subs([(x, 0), (y, 0)])
print(f"\nH at (0,0):\n{H_origin}")

H_np = np.array(H_origin.tolist(), dtype=float)
eigenvalues = np.linalg.eigvalsh(H_np)
print(f"Eigenvalues: {eigenvalues}")
print(f"Classification: {'Saddle point' if eigenvalues[0] * eigenvalues[-1] < 0 else 'Min/Max'}")

# Visualize saddle point
fig = plt.figure(figsize=(12, 5))

# 3D surface of a saddle: f(x,y) = x² - y²
f_saddle = lambda x, y: x**2 - y**2
X, Y = np.meshgrid(np.linspace(-2, 2, 50), np.linspace(-2, 2, 50))

ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X, Y, f_saddle(X, Y), cmap='coolwarm', alpha=0.8)
ax1.set_title('Saddle Point: f = x² - y²', fontsize=14)

# Bowl (minimum)
f_bowl = lambda x, y: x**2 + y**2
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(X, Y, f_bowl(X, Y), cmap='viridis', alpha=0.8)
ax2.set_title('Minimum: f = x² + y²', fontsize=14)

plt.tight_layout()
plt.show()

---
## 6. Multivariable Chain Rule

If $f$ depends on $\mathbf{g}(t)$ which depends on $t$:

$$\frac{df}{dt} = \sum_{i} \frac{\partial f}{\partial g_i} \frac{dg_i}{dt} = \nabla f \cdot \frac{d\mathbf{g}}{dt}$$

### General form for composition $f \circ \mathbf{g}$:
$$\frac{\partial f}{\partial x_j} = \sum_{i} \frac{\partial f}{\partial g_i} \frac{\partial g_i}{\partial x_j}$$

In matrix form: the Jacobian of the composition is the **product of Jacobians**!

$$J_{f \circ g} = J_f \cdot J_g$$

### 🤖 This IS backpropagation!
In a 3-layer network: $L = f_3(f_2(f_1(x)))$

$$\frac{\partial L}{\partial x} = J_{f_3} \cdot J_{f_2} \cdot J_{f_1}$$

In [ ]:
# Multivariable chain rule = Backpropagation!
print("=== Backpropagation as Chain Rule ===")

# Simple 2-layer network: y = σ(W₂ · σ(W₁ · x))
np.random.seed(42)

# Forward pass
x_input = np.array([1.0, 2.0])  # Input
W1 = np.array([[0.5, -0.3], [0.2, 0.8]])  # Layer 1 weights
W2 = np.array([[0.4, 0.6]])  # Layer 2 weights

# Forward
z1 = W1 @ x_input            # Linear 1
a1 = np.maximum(0, z1)       # ReLU
z2 = W2 @ a1                 # Linear 2
y_pred = z2[0]               # Output
target = 1.0
loss = (y_pred - target)**2  # MSE loss

print(f"Forward: x={x_input} → z1={z1} → a1={a1} → z2={z2} → loss={loss:.4f}")

# Backward pass (chain rule!)
dL_dy = 2 * (y_pred - target)        # ∂L/∂y
dy_dz2 = 1                            # ∂y/∂z₂
dz2_da1 = W2                          # ∂z₂/∂a₁ = W₂
da1_dz1 = (z1 > 0).astype(float)     # ∂a₁/∂z₁ (ReLU derivative)
dz1_dW1 = x_input                    # ∂z₁/∂W₁

# Chain rule: ∂L/∂W₂
dL_dW2 = dL_dy * dy_dz2 * a1  # = dL/dy · dy/dz₂ · ∂z₂/∂W₂
print(f"\n∂L/∂W₂ = {dL_dW2}")

# Chain rule: ∂L/∂W₁ (goes through more layers)
dL_da1 = dL_dy * W2[0]  # Backprop through layer 2
dL_dz1 = dL_da1 * da1_dz1  # Through ReLU
dL_dW1 = np.outer(dL_dz1, x_input)  # Through layer 1
print(f"∂L/∂W₁ = \n{np.round(dL_dW1, 4)}")
print("\n✅ This is exactly what PyTorch's autograd does!")

---
## 7. Lagrange Multipliers — Constrained Optimization

**Problem**: Minimize $f(x, y)$ subject to constraint $g(x, y) = 0$.

**Method**: At the optimum, the gradients must be parallel:
$$\nabla f = \lambda \nabla g$$

Solve the system:
$$\frac{\partial f}{\partial x} = \lambda \frac{\partial g}{\partial x}$$
$$\frac{\partial f}{\partial y} = \lambda \frac{\partial g}{\partial y}$$
$$g(x, y) = 0$$

### 🤖 AI Connection:
- **SVMs**: Maximize margin subject to classification constraints
- **KKT conditions** generalize Lagrange multipliers to inequality constraints
- **Regularization** can be viewed as constrained optimization

In [ ]:
# Lagrange multipliers example:
# Minimize f(x,y) = x + y subject to x² + y² = 1 (on unit circle)

lam = sp.Symbol('lambda')
f = x + y
g = x**2 + y**2 - 1

# Lagrangian: L = f - λg
L = f - lam * g

# Solve ∂L/∂x = 0, ∂L/∂y = 0, g = 0
equations = [sp.diff(L, x), sp.diff(L, y), g]
solutions = sp.solve(equations, [x, y, lam])

print("Minimize f(x,y) = x + y on the unit circle x² + y² = 1")
print(f"\nSolutions:")
for sol in solutions:
    val = f.subs([(x, sol[0]), (y, sol[1])])
    print(f"  ({sol[0]}, {sol[1]}): f = {val}, λ = {sol[2]}")

# Visualize
fig, ax = plt.subplots(figsize=(8, 8))
theta = np.linspace(0, 2*np.pi, 100)
ax.plot(np.cos(theta), np.sin(theta), 'b-', linewidth=2, label='x² + y² = 1')

# Level curves of f = x + y
X, Y = np.meshgrid(np.linspace(-1.5, 1.5, 100), np.linspace(-1.5, 1.5, 100))
cs = ax.contour(X, Y, X + Y, levels=10, cmap='RdYlGn', alpha=0.5)
ax.clabel(cs, inline=True, fontsize=8)

for sol in solutions:
    sx, sy = float(sol[0]), float(sol[1])
    val = float(f.subs([(x, sol[0]), (y, sol[1])]))
    ax.plot(sx, sy, 'ro', markersize=10)
    ax.annotate(f'f={val:.2f}', xy=(sx, sy), xytext=(sx+0.2, sy+0.2), fontsize=12)

ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title('Lagrange Multipliers: Optimize f=x+y on Unit Circle', fontsize=14)
ax.legend()
plt.show()

---
## 8. Multivariable Gradient Descent

Now with multiple variables:
$$\boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \alpha \nabla f(\boldsymbol{\theta}_t)$$

Let's optimize the **Rosenbrock function** — a classic hard optimization benchmark:
$$f(x, y) = (1-x)^2 + 100(y - x^2)^2$$

Minimum at $(1, 1)$ but the valley is very narrow!

In [ ]:
# 2D Gradient Descent on Rosenbrock function
def rosenbrock(p):
    return (1 - p[0])**2 + 100*(p[1] - p[0]**2)**2

def rosenbrock_grad(p):
    dx = -2*(1-p[0]) - 400*p[0]*(p[1]-p[0]**2)
    dy = 200*(p[1]-p[0]**2)
    return np.array([dx, dy])

# Run gradient descent
p = np.array([-1.0, 1.0])
lr = 0.001
path = [p.copy()]

for _ in range(5000):
    grad = rosenbrock_grad(p)
    p = p - lr * grad
    path.append(p.copy())

path = np.array(path)
print(f"Start: ({path[0][0]:.2f}, {path[0][1]:.2f}), f = {rosenbrock(path[0]):.2f}")
print(f"End: ({path[-1][0]:.4f}, {path[-1][1]:.4f}), f = {rosenbrock(path[-1]):.6f}")

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
X, Y = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-1, 3, 200))
Z = (1-X)**2 + 100*(Y-X**2)**2

ax.contour(X, Y, np.log(Z + 1), levels=30, cmap='viridis')
ax.plot(path[:, 0], path[:, 1], 'r.-', markersize=1, linewidth=0.5, alpha=0.7)
ax.plot(path[0, 0], path[0, 1], 'g*', markersize=15, label='Start')
ax.plot(1, 1, 'r*', markersize=15, label='Minimum (1,1)')
ax.set_title('Gradient Descent on Rosenbrock Function', fontsize=16)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)
plt.show()

---
## 9. Multiple Integrals

### Double Integral
$$\iint_R f(x,y) \, dA = \int_a^b \int_c^d f(x,y) \, dy \, dx$$

### 🤖 AI Connection:
- **Marginalizing** over a variable: $p(x) = \int p(x, y) \, dy$
- **Evidence computation** in Bayesian inference
- Computing **expected values** in 2D

In [ ]:
# Double integral
f = x**2 + y**2
result = sp.integrate(f, (y, 0, 1), (x, 0, 1))
print(f"∫∫ (x²+y²) dA over [0,1]×[0,1] = {result}")

# 2D Gaussian normalization check
# ∫∫ exp(-(x²+y²)/2) dx dy = 2π
result = sp.integrate(sp.exp(-(x**2 + y**2)/2), (x, -sp.oo, sp.oo), (y, -sp.oo, sp.oo))
print(f"\n∫∫ exp(-(x²+y²)/2) dxdy = {result} = 2π")

# Numerical double integral
from scipy import integrate as sci_int
result_num, error = sci_int.dblquad(lambda y, x: np.exp(-(x**2+y**2)/2), -5, 5, -5, 5)
print(f"Numerical: {result_num:.6f} (2π = {2*np.pi:.6f})")

---
## 10. Summary: Multivariable Calculus for AI

| Concept | Formula | AI Application |
|---------|---------|---------------|
| Gradient | $\nabla f$ | Gradient descent direction |
| Jacobian | $J_{ij} = \frac{\partial f_i}{\partial x_j}$ | Backpropagation, normalizing flows |
| Hessian | $H_{ij} = \frac{\partial^2 f}{\partial x_i \partial x_j}$ | Newton's method, loss curvature |
| Chain Rule | $J_{f \circ g} = J_f \cdot J_g$ | Backpropagation |
| Lagrange | $\nabla f = \lambda \nabla g$ | SVMs, constrained optimization |
| Multiple Integrals | $\iint f \, dA$ | Marginalization, expectations |

**You now have the complete calculus toolkit! Next: Probability Theory — the language of uncertainty in AI.** 🚀